# FloodDuo — flood mapping inference

Run the released FloodDuo model on a PlanetScope-style tile and get a binary
water/flood map. Everything (weights + a sample tile) downloads automatically
when you run the cells below — no manual setup beyond installing packages.

This uses the **release checkpoint** (trained on all FloodPlanet + UFO data
pooled together), not one of the paper's leave-one-region-out fold models —
see the note on the [Hugging Face model card](https://huggingface.co/Sk-21/FloodDuo)
for why that distinction matters.

Works on CPU (slower) or GPU; no GPU required for a single tile.

## 1. Install dependencies

In [ ]:
%pip install -q torch torchvision timm rasterio einops huggingface_hub matplotlib numpy

## 2. Download the checkpoint and a sample tile

`floodduo_full.pt` (~1.3 GB) is self-contained: both frozen encoder backbones
(DINOv3-L, Clay v1.5) and FloodDuo's trained adapters/fusion/decoder are all
in this one file, so this is the only download needed to run inference.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, for `import src`

from huggingface_hub import hf_hub_download

REPO = "Sk-21/FloodDuo"
ckpt_path = hf_hub_download(REPO, "floodduo_full.pt")
sample_image_path = hf_hub_download(REPO, "examples/sample_tile.tif")
sample_mask_path = hf_hub_download(REPO, "examples/sample_mask.tif")
print("downloaded:", ckpt_path)

## 3. Load the model

In [ ]:
import torch
from src.inference import load_model, predict_file, save_mask

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model(ckpt_path, device=device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"loaded on {device}, {n_params:.0f}M parameters")

## 4. Run inference

Input is raw surface reflectance, 4 bands (B, G, R, NIR), float32 — no manual
normalization needed, the model handles that internally.

In [ ]:
prob, mask, profile = predict_file(model, sample_image_path, device=device)
print(f"predicted water fraction: {mask.mean():.1%}")

## 5. Visualize

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(sample_image_path) as src:
    img = src.read()  # (4, H, W) = B, G, R, NIR
with rasterio.open(sample_mask_path) as src:
    gt = src.read(1) > 0.5


def false_color(img):
    x = img[[3, 2, 1]].astype(float)  # NIR, R, G
    out = np.empty_like(x)
    for k in range(3):
        lo, hi = np.percentile(x[k], 2), np.percentile(x[k], 98)
        out[k] = np.clip((x[k] - lo) / (hi - lo + 1e-6), 0, 1)
    return np.transpose(out, (1, 2, 0))


fig, ax = plt.subplots(1, 4, figsize=(18, 5))
ax[0].imshow(false_color(img)); ax[0].set_title("False color (NIR-R-G)")
ax[1].imshow(gt, cmap="Blues", vmin=0, vmax=1); ax[1].set_title("Ground truth")
ax[2].imshow(mask, cmap="Blues", vmin=0, vmax=1); ax[2].set_title("FloodDuo prediction")
ax[3].imshow(false_color(img)); ax[3].imshow(mask, cmap="Blues", alpha=0.45)
ax[3].set_title("Prediction overlay")
for a in ax: a.set_xticks([]); a.set_yticks([])
fig.tight_layout()

## 6. Use your own image

Point `my_image_path` at any 4-band (B, G, R, NIR) surface-reflectance
GeoTIFF — PlanetScope or similar, float32, values roughly in [0, 0.5+] — and
rerun from here. Tiles are evaluated at their native size; for a large scene,
chip it into ~512-1024 px tiles first (the model wasn't evaluated at larger
sizes in the paper).

In [ ]:
my_image_path = sample_image_path  # <- replace with your own .tif path

prob, mask, profile = predict_file(model, my_image_path, device=device)
plt.figure(figsize=(5, 5))
plt.imshow(mask, cmap="Blues", vmin=0, vmax=1)
plt.title(f"Predicted water fraction: {mask.mean():.1%}")
plt.xticks([]); plt.yticks([])

## 7. Save the predicted mask as a GeoTIFF

In [ ]:
out_path = "floodduo_prediction.tif"
save_mask(mask, profile, out_path)
print("saved:", out_path)